# NonlinearOptimizerParams

`NonlinearOptimizerParams` holds convergence, iteration, ordering, verbosity, and linear-solver settings shared by GTSAM's batch nonlinear optimizers. Concrete optimizer parameter classes inherit this interface.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/nonlinear/doc/NonlinearOptimizerParams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np
from gtsam.symbol_shorthand import L, V, X

## Creating and configuring the common parameters

The base object is constructible for inspection, but applications normally create `GaussNewtonParams`, `DoglegParams`, or `LevenbergMarquardtParams` and use the inherited setters.

In [ ]:
params = gtsam.GaussNewtonParams()
params.setMaxIterations(50)
params.setRelativeErrorTol(1e-6)
params.setAbsoluteErrorTol(1e-8)
params.setErrorTol(0.0)
params.setVerbosity("SILENT")
print("maximum iterations:", params.getMaxIterations())

## Linear solver and ordering

`setLinearSolverType()` accepts names such as `MULTIFRONTAL_CHOLESKY` and `SEQUENTIAL_QR`. Query methods identify the selected solver family. An explicit `Ordering` can override the automatic ordering for reproducibility or known sparsity structure.

In [ ]:
params.setLinearSolverType("MULTIFRONTAL_CHOLESKY")
ordering = gtsam.Ordering()
ordering.push_back(X(0))
params.setOrdering(ordering)
print("solver:", params.getLinearSolverType())
print("is multifrontal:", params.isMultifrontal())

## Iteration hooks

Python can assign `iterationHook` to receive `(iteration, error_before, error_after)` after each iteration. This is useful for logging without increasing optimizer verbosity.

In [ ]:
history = []
params.iterationHook = lambda iteration, before, after: history.append(
    (iteration, before, after)
)

graph = gtsam.NonlinearFactorGraph()
model = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.05]))
graph.add(gtsam.PriorFactorPose2(X(0), gtsam.Pose2(1.0, 0.0, 0.2), model))
initial = gtsam.Values()
initial.insert(X(0), gtsam.Pose2())
gtsam.GaussNewtonOptimizer(graph, initial, params).optimize()
print("iterations recorded:", len(history))

## Source

[`NonlinearOptimizerParams.h`](../NonlinearOptimizerParams.h)

## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.